# Serve + Test the E4B adapter on Colab T4 (HF nf4 — full fidelity)

Serves your trained E4B LoRA at the SAME nf4 4-bit precision it was trained on (no GGUF
quant degradation), on a free Colab **T4 (16GB)** which fits E4B 4-bit with room. This is
the truest read of model quality — q4_0 GGUF would confound it (and we've seen q4_0
fabricate numbers).

**Two things:** Cell 7 runs my probe prompts (you read the raw behavior); Cell 8 serves the
web UI via an ngrok tunnel so YOU can chat-test in the browser.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Add Colab Secrets:
`HF_TOKEN` (HF read) and `NGROK_TOKEN` (free from ngrok.com → authtoken). Put your
downloaded `best/` adapter folder in Google Drive (e.g. `MyDrive/chess_adapter/best`).

In [ ]:
# Cell 1 - config
import os
BASE_REPO = "unsloth/gemma-4-E4B-it"   # the base your adapter was trained on
REPO_URL  = "https://github.com/RyanDev1st/llm-and-engine.git"
BRANCH    = "feat/chess-coach-sft"

if os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    WORKDIR = "A:/Download/llm_tool_calling_research_workspace"
    ADAPTER_DIR = "A:/Download/gemma4_chess_e4b_kaggle (1)"
else:
    WORKDIR = "/content/llm-and-engine"
    ADAPTER_DIR = "/content/adapter"

BASE_DIR  = f"{WORKDIR}/src/llm/models/gemma4_e4b"
print("base:", BASE_REPO, "| adapter:", ADAPTER_DIR)

In [ ]:
# Cell 2 - GPU check (want a T4 16GB)
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,memory.free","--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 3 - clone repo
import os, subprocess
env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(["git","-C",WORKDIR,"pull","--ff-only"], check=True, env=env)
subprocess.run(["git","-C",WORKDIR,"log","-1","--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 - deps (serve path + ngrok + stockfish for full chess tools)
import subprocess, sys, shutil
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
pip("-U","transformers","accelerate","peft","bitsandbytes","sentencepiece","protobuf","python-chess","pyngrok")
if shutil.which("apt-get"):
    subprocess.run(["apt-get","-qq","install","-y","stockfish"])  # chess tools (optional)
import transformers, peft, bitsandbytes
print("transformers",transformers.__version__,"| peft",peft.__version__,"| bnb",bitsandbytes.__version__)

In [ ]:
# Cell 5 - HF login + download the E4B base (~9GB; once)
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    login(os.environ["HF_TOKEN"])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=["*.json","*.safetensors","*.model","*.txt","tokenizer*"])
print("base at", BASE_DIR, "->", sorted(os.listdir(BASE_DIR))[:8])

In [ ]:
# Cell 6 - download the adapter from Hugging Face
import os
from huggingface_hub import snapshot_download

ADAPTER_REPO = "RyanDev1st/gemma4-chesscoach-ckpt"

if not os.path.exists("A:/Download/llm_tool_calling_research_workspace"):
    print(f"Downloading adapter from HF: {ADAPTER_REPO}")
    snapshot_download(repo_id=ADAPTER_REPO, local_dir="/content/hf_adapter",
                      allow_patterns=["best/*"])
    ADAPTER_DIR = "/content/hf_adapter/best"

need = ("adapter_model.safetensors","adapter_config.json")
ok = all(os.path.exists(f"{ADAPTER_DIR}/{f}") for f in need)
if os.path.exists(ADAPTER_DIR):
    print("adapter at", ADAPTER_DIR, "->", os.listdir(ADAPTER_DIR))
else:
    print("adapter at", ADAPTER_DIR, "-> NOT FOUND")
assert ok, f"adapter not found at {ADAPTER_DIR}"

In [ ]:
# Cell 6b - (alt) upload the adapter instead of Drive. Zip best/ locally, upload here.
# from google.colab import files; import zipfile, io, os
# up = files.upload()                       # pick best.zip
# name = next(iter(up))
# ADAPTER_DIR = "/content/best"
# with zipfile.ZipFile(io.BytesIO(up[name])) as z: z.extractall("/content")
# print(os.listdir(ADAPTER_DIR))

In [ ]:
# Cell 7 - PROBE via the model SERVICE (ONE model, shared with the web UI; no double-load).
# The model loads ONCE inside backend.model_server; this cell + Cell 8 both talk to it over
# HTTP, so re-running never creates a 2nd GPU copy (that was the OOM).
import os, sys, subprocess, time, json, urllib.request
os.environ['CHESS_HF_BASE'] = BASE_DIR
SVC = 'http://127.0.0.1:7861'

def _health():
    try: return json.load(urllib.request.urlopen(SVC+'/health', timeout=5)).get('ok')
    except Exception: return False

def ensure_service():
    if _health():
        print('model service already UP - reusing (no reload)'); return
    subprocess.run(['pkill','-f','backend.model_server']); time.sleep(2)
    env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR}
    log=open('/content/model_server.log','w')
    globals()['_MSVC']=subprocess.Popen([sys.executable,'-u','-m','backend.model_server',ADAPTER_DIR],
                                        cwd=f'{WORKDIR}/src/llm',env=env,stdout=log,stderr=subprocess.STDOUT)
    print('starting model service (loads weights ONCE, ~1-2 min)...')
    for _ in range(180):
        time.sleep(2)
        if globals()['_MSVC'].poll() is not None:
            print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service crashed')
        if _health(): print('model service UP'); return
    print(open('/content/model_server.log').read()[-3000:]); raise RuntimeError('service did not come up')

ensure_service()
sys.path.insert(0, f'{WORKDIR}/src/llm')
from backend.inference import build_system_prompt

def step(prompt, mode, stop=('</tool>','</skill>'), mx=220):
    sysp = build_system_prompt(reasoning_mode=mode)
    body = json.dumps({'messages':[{'role':'system','content':sysp},{'role':'user','content':prompt}],
                       'max_new_tokens':mx,'stop':list(stop),'use_adapter':True}).encode()
    req = urllib.request.Request(SVC+'/generate', data=body, headers={'content-type':'application/json'})
    out = json.load(urllib.request.urlopen(req, timeout=180))['text']
    print(f'### [{mode}] {prompt}')
    print(out)
    print('-'*70)

step('I scored 70, 10, and 8 - is my average above 85?', 'think')
step('What is a 20% tip on a $138.60 bill?', 'auto')
step('Can you review this SQL query for bugs and tell me why it is slow?', 'think')
step('What is the best move here?', 'auto')
step('hey, what can you do?', 'fast')
step('hey, what can you do?', 'think')


In [ ]:
# Cell 8 - SERVE the web UI (reuses the SAME model service from Cell 7; ngrok tunnel).
# Only the weightless APP restarts here - the model is NOT reloaded (no OOM).
import os, sys, subprocess, time
ensure_service()                      # reuse the one model; starts it only if not already up
env={**os.environ,'PYTHONPATH':f'{WORKDIR}/src/llm','CHESS_HF_ADAPTER':ADAPTER_DIR,'CHESS_HF_BASE':BASE_DIR,
     'CHESS_MODEL_SERVER':SVC,'CHESS_HOST':'127.0.0.1','CHESS_PORT':'7860'}
subprocess.run(['pkill','-f','backend.server']); time.sleep(2)   # restart app only (cheap, weightless)
globals()['_APP']=subprocess.Popen([sys.executable,'-u','-m','backend.server'],cwd=f'{WORKDIR}/src/llm',
                  env=env,stdout=open('/content/app_server.log','w'),stderr=subprocess.STDOUT)
time.sleep(5)
from pyngrok import ngrok
ngrok.kill()                          # drop old tunnels (free tier allows 1)
try:
    from google.colab import userdata; ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
except Exception:
    ngrok.set_auth_token(os.environ['NGROK_TOKEN'])
url = ngrok.connect(7860)
print('==============================================')
print('  OPEN THIS IN YOUR BROWSER TO CHAT-TEST:')
print('  ', url)
print('==============================================')
print('--- app server log ---')
print(open('/content/app_server.log').read()[-1500:])
